# Distributor RFQ → Item Matching Prototype

Takes unstructured RFQ text and matches each part number against the IMS catalog.

**Output per line:** `Exact` (ready to quote) · `Cross Ref` (OEM→DST) · `Partial` (similar, needs review) · `No Match` (manual review)

**Smart parsing:** When the RFQ says *"X equivalent to Y"*, Y is captured as a **Customer Reference** linked to X (not a separate row).

**How to run:** Run cells top-to-bottom. When prompted in Cell 2, upload `IMS_Items_Exp.csv`.

## 1. Imports

In [ ]:
import re
import pandas as pd
from difflib import get_close_matches

## 2. Upload the catalog CSV

When the upload widget appears, select **`IMS_Items_Exp.csv`** from your computer.

In [ ]:
from google.colab import files
uploaded = files.upload()
CATALOG_PATH = list(uploaded.keys())[0]
print(f"Loaded: {CATALOG_PATH}")

## 3. Cross-reference map

Hardcoded OEM → DST mappings, taken from real Distributor Estimate #10001.

In [ ]:
OEM_TO_DST = {
    "OEM9087L": "DST-0615L",
    "OEM9087R": "DST-0615R",
    "OEM9110L": "DST-0616L",
    "OEM9110R": "DST-0616R",
}

## 4. The 4 RFQ examples

Edit any of these strings to test a new RFQ — then re-run cells 6 and 7.

In [ ]:
RFQS = {
    "RFQ 1": """Can you please help me out with a sales quote on the following:
PN: 2-164-551 BLOCK, REFERENCE STANDARD
QTY: 1""",

    "RFQ 2": """Please submit your Bids for the items listed below:
PartNumber Description Condition Qty/UM
SC-BNC-UHF-6  BNC to UHF CABLE FOR ULTRASONIC TRANSDUCER  2 EA""",

    "RFQ 3": """Flexible roto probe
DS-5007/7.8MM equivalent to KD1H52408-2
QTY: 1
Specs: Frequency 100KHZ-2MHZ, Diameter 7.8 mm, working length 90mm

Flexible roto probe
DS-5007/6.1MM equivalent to KD1H52404-2
QTY: 1
Specs: Frequency 100KHZ-2MHZ, Diameter 6.1 mm, working length 90mm""",

    "RFQ 4": """Please quote the following PN:
NDT3317
NDT3318
NDT3310-A
NDT3310-B
NDT3310-C
OEM-9087L
OEM-9087-R
OEM-9110L
OEM-9110R""",
}

## 5. Matching engine

Helpers:
1. **`normalize`** — uppercase + strip spaces/dashes/punctuation so `OEM-9087-R` and `OEM9087R` match.
2. **`extract_rfq_items`** — pulls part-number tokens out of messy RFQ text. Detects `"X equivalent to Y"` and links Y as a Customer Reference of X (Y is *not* a separate row).
3. **`match_one`** — the cascade: Exact → Cross Ref → Partial → No Match.

In [ ]:
def normalize(s):
    if s is None: return ""
    return re.sub(r"[\s\-_/.,:]+", "", str(s)).upper()

NOISE = {
    "PN", "QTY", "EA", "UM", "CONDITION", "DESCRIPTION", "PART", "NUMBER",
    "PARTNUMBER", "REMARKS", "ITEM", "BIDS", "BID", "RFQ", "PLEASE",
    "EQUIVALENT", "FREQUENCY", "DIAMETER", "WORKING", "LENGTH", "TIP",
    "COIL", "DISTANCE", "MM", "KHZ", "MHZ", "TO", "OF", "C", "TDS",
    "TEST", "REPORT", "FLEXIBLE", "ROTO", "PROBE", "SUBMIT", "FOR",
    "FROM", "AND", "THE", "WITH", "BLOCK", "REFERENCE", "STANDARD",
    "BNC", "UHF", "CABLE", "ULTRASONIC", "TRANSDUCER",
}

UNIT_RE = re.compile(
    r"^\d+(\.\d+)?(MM|CM|IN|HZ|KHZ|MHZ|GHZ|EA|PCS|G|KG|LB|OZ|V|W|MA)"
    r"(-\d+(\.\d+)?(MM|CM|IN|HZ|KHZ|MHZ|GHZ|EA|PCS|G|KG|LB|OZ|V|W|MA))?$"
)

PN_TOKEN = r"[A-Za-z0-9][A-Za-z0-9\-_/\.]{3,}"
EQUIV_RE = re.compile(
    rf"({PN_TOKEN})\s+(?:equivalent\s+to|equivalent|equiv\.?\s*to|eq\.?\s*to)\s+({PN_TOKEN})",
    re.IGNORECASE,
)

def _valid_pn(u):
    if u in NOISE: return False
    if not re.search(r"\d", u): return False
    if re.fullmatch(r"\d+(\.\d+)?", u): return False
    if UNIT_RE.match(u): return False
    return True

def extract_rfq_items(text):
    """Return list of {'pn', 'customer_ref'}. Links 'X equivalent to Y' as customer_ref."""
    primary_to_ref, skip_norm = {}, set()
    for m in EQUIV_RE.finditer(text):
        primary, equiv = m.group(1), m.group(2)
        if not _valid_pn(primary.upper().strip(".,:")) or not _valid_pn(equiv.upper().strip(".,:")):
            continue
        primary_to_ref[normalize(primary)] = equiv
        skip_norm.add(normalize(equiv))

    tokens = re.findall(r"[A-Za-z0-9][A-Za-z0-9\-_/\.]{3,}", text)
    seen, out = set(), []
    for t in tokens:
        u = t.upper().strip(".,:")
        if not _valid_pn(u): continue
        n = normalize(t)
        if n in skip_norm or n in seen: continue
        seen.add(n)
        out.append({"pn": t, "customer_ref": primary_to_ref.get(n)})
    return out

def match_one(rfq_pn, catalog_df, norm_index, norm_list):
    n = normalize(rfq_pn)

    if n in norm_index:
        row = catalog_df.iloc[norm_index[n]]
        return {"Match Type": "Exact", "Internal Part Number": row["Item Number"],
                "Confidence": "High", "Notes": f"Exact catalog match ({row['Item Name']})"}

    if n in OEM_TO_DST:
        return {"Match Type": "Cross Ref", "Internal Part Number": OEM_TO_DST[n],
                "Confidence": "High", "Notes": "Mapped from OEM to DST (per Estimate #10001)"}

    candidates = get_close_matches(n, norm_list, n=2, cutoff=0.6)
    if not candidates and len(n) >= 4:
        candidates = [x for x in norm_list if n in x or x in n][:2]
    if candidates:
        picks = [f"{catalog_df.iloc[norm_index[c]]['Item Number']} ({catalog_df.iloc[norm_index[c]]['Item Name']})" for c in candidates]
        return {"Match Type": "Partial",
                "Internal Part Number": "; ".join(catalog_df.iloc[norm_index[c]]["Item Number"] for c in candidates),
                "Confidence": "Medium",
                "Notes": "Similar item, needs review: " + " | ".join(picks)}

    return {"Match Type": "No Match", "Internal Part Number": "",
            "Confidence": "Low", "Notes": "Not found in catalog - manual review required"}

## 6. Run the pipeline & display results

In [ ]:
cat = pd.read_csv(CATALOG_PATH, skiprows=2)
cat = cat.dropna(subset=["Item Number"]).reset_index(drop=True)
print(f"Catalog loaded: {len(cat)} items\n")

norm_index = {}
for i, pn in enumerate(cat["Item Number"]):
    n = normalize(pn)
    if n and n not in norm_index:
        norm_index[n] = i
norm_list = list(norm_index.keys())

rows = []
for label, text in RFQS.items():
    for item in extract_rfq_items(text):
        pn, cust_ref = item["pn"], item["customer_ref"]
        r = match_one(pn, cat, norm_index, norm_list)
        if cust_ref:
            r["Notes"] = f"{r['Notes']}; customer also referenced equivalent '{cust_ref}'"
        rows.append({"Source RFQ": label, "RFQ Item": pn,
                     "Customer Reference": cust_ref or "", **r})

result = pd.DataFrame(rows, columns=[
    "Source RFQ", "RFQ Item", "Customer Reference", "Match Type",
    "Internal Part Number", "Confidence", "Notes"
])

def _color(row):
    c = {"Exact": "#d4edda", "Cross Ref": "#cce5ff",
         "Partial": "#fff3cd", "No Match": "#f8d7da"}.get(row["Match Type"], "")
    return [f"background-color: {c}"] * len(row)

result.style.apply(_color, axis=1)

## 7. Summary counts

In [ ]:
summary = result["Match Type"].value_counts().to_frame("Count")
print(summary.to_string())
print(f"\nTotal RFQ items processed: {len(result)}")

## 8. Download results as CSV

In [ ]:
OUT_PATH = "rfq_match_output.csv"
result.to_csv(OUT_PATH, index=False)
from google.colab import files
files.download(OUT_PATH)